# Stage 5M: Closed-Loop Online Control Suite
## Controlled Research Evaluation of the Operational Disparity Safety Gate (DSG)
### Platform: Kaggle Notebooks (NVIDIA Tesla P100 GPU / 16GB VRAM)

**Protocol**: `v1.1.0-model-expansion` | **Standard**: ADR-005  
**Stakeholder Review Status**: **CONDITIONAL GO** — Authorized for Controlled Research Evaluation

---

### Scientific Purpose & Invariants
In Stage 4M, retrospective auditing revealed that **85.2%** of unconstrained test-time adaptation settings failed at least one preregistered safety criterion.

**Stage 5M** deploys the operational **Disparity Safety Gate (DSG)** online during sequential streaming across 5 acoustic conditions to determine whether sentinel-controlled gating intercepts harmful updates:
1. **Frozen Safety Policy**:
   $$\epsilon_R = 0.00\text{ pp} \ (0.0000), \quad \epsilon_G = +2.00\text{ pp} \ (0.0200), \quad \epsilon_D = +2.00\text{ pp} \ (0.0200)$$
   $$B = 1,000 \text{ resamples}, \quad \alpha = 0.05 \ (95\% \text{ confidence})$$
2. **Strict Operational Decoupling**:
   - Candidate update decisions are evaluated exclusively on the **Sentinel Panel ($N=30$ independent speakers, 300 clips)**.
   - Test-stream reference transcripts remain strictly quarantined from the online controller via `LabelIsolationSanitizer`.
3. **Primary Diagnostic Result — Operational vs. Retrospective Agreement**:
   - Compares the operational sentinel decision against the retrospective ground-truth outcome on the test stream.
   - Specifically tracks **False Approvals** (cases where the gate accepted an update that was retrospectively harmful on the test stream).
4. **Preregistered Model Roles**:
   - **CTC Qualification Backbones**: `wav2vec2_base`, `data2vec_base`, `wav2vec2_100h`.
   - **Seq2Seq Static Portability Controls**: `whisper_base`, `distil_whisper_small`, `whisper_tiny` evaluated strictly as static No-Adapt controls.
5. **Boundary**: Stage 6M remains unauthorized until these Stage 5M results are audited.

---

### Kaggle Setup Instructions:
1. In the right-hand panel under **Notebook options**:
   - **Accelerator**: Select **GPU P100**.
   - **Internet**: Toggle **ON**.
2. Run all cells sequentially from top to bottom.

In [ ]:
# Step 1: Verify Hardware Accelerator (NVIDIA Tesla P100 / 16GB VRAM)
import torch
print("=" * 75)
print("STEP 1: HARDWARE ACCELERATOR VERIFICATION")
print("=" * 75)
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device     : {gpu_name}")
    print(f"Total VRAM     : {vram_gb:.2f} GB")
    if "P100" in gpu_name or "T4" in gpu_name or vram_gb >= 14.0:
        print("Assessment     : EXCELLENT (High-speed 16GB GPU ready for batched sentinel inference)")
else:
    print("WARNING: No GPU detected! Please enable GPU in Kaggle settings (Accelerator -> GPU P100).")

In [ ]:
# Step 2: Environment Setup & Audio Dataset Extraction
# Automatically installs dependencies, unpacks stage5m_audio_bundle.tar.gz,
# and runs Stage 5M research validity integrity tests.
!python cloud/kaggle/setup_kaggle.py

In [ ]:
# Step 3: Fast Preflight Smoke Test (Validates Sentinel Gate in ~30 seconds)
# Runs 2 windows on wav2vec2_base with paired bootstrap (B=1000) and label isolation firewall.
!python cloud/kaggle/run_stage5m_online_control.py --smoke-test

In [ ]:
# Step 4: Execute Full Stage 5M Closed-Loop Online Control Suite
# Evaluates 3 CTC backbones x 5 acoustic conditions x 3 adaptation methods x 3 stream orders
# plus 3 Seq2Seq static portability controls under frozen DSG control.
!python cloud/kaggle/run_stage5m_online_control.py

In [ ]:
# Step 5: Display Closed-Loop Recognition Outcomes Across Acoustic Stress Conditions
import os
import pandas as pd
from IPython.display import display

res_path = "results/stage5m/stage5m_closed_loop_results.csv"
if os.path.exists(res_path):
    df_res = pd.read_csv(res_path)
    print("=" * 80)
    print("STAGE 5M: CLOSED-LOOP RECOGNITION OUTCOMES (WER & DISPARITY)")
    print("=" * 80)
    cols = ["model_key", "condition", "method", "ordering_id", "final_wer", "final_disparity", "acceptance_rate", "false_approvals"]
    display(df_res[cols])
else:
    print(f"Results file not found: {res_path}")

In [ ]:
# Step 6: Display Primary Result: Operational Decisions vs. Retrospective Ground-Truth
# Mandated by Condition C of Stakeholder Review: Tracks False Approvals vs True Interceptions
agree_path = "results/stage5m/stage5m_decision_agreement_matrix.csv"
if os.path.exists(agree_path):
    print("=" * 80)
    print("STAGE 5M: OPERATIONAL DECISIONS VS. RETROSPECTIVE GROUND-TRUTH MATRIX")
    print("=" * 80)
    df_agree = pd.read_csv(agree_path, index_col=0)
    display(df_agree)
else:
    print(f"Agreement matrix not found: {agree_path}")

In [ ]:
# Step 7: Display Markdown Report Summary
report_path = "reports/stage5m/stage5m_closed_loop_report.md"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        content = f.read()
    print("=" * 80)
    print("STAGE 5M OFFICIAL CLOSED-LOOP REPORT PREVIEW")
    print("=" * 80)
    print(content[:2500])
    if len(content) > 2500:
        print("\n... [truncated preview, see full report in reports/stage5m/stage5m_closed_loop_report.md]")

In [ ]:
# Step 8: Package Results into Ready-to-Download Archive
!python cloud/kaggle/export_results.py

In [ ]:
# Step 9: Verify Download Archive in /kaggle/working/
import os
bundle_5m = "/kaggle/working/stage5m_results_bundle.zip"
if not os.path.exists(bundle_5m):
    bundle_5m = "reports/stage5m_results_bundle.zip"

if os.path.exists(bundle_5m):
    sz_kb = os.path.getsize(bundle_5m) / 1024
    print("=" * 80)
    print(f"SUCCESS: Stage 5M results bundle ready for download ({sz_kb:.1f} KB)")
    print(f"Location: {bundle_5m}")
    print("=" * 80)
    print("\n--> In Kaggle: Look in the right-hand panel under 'Output' -> Click the three dots next to 'stage5m_results_bundle.zip' -> Download.")
else:
    print("Notice: Check reports/ directory for generated zip files.")